# Time-Series Forecasting: Electricity Production

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CDAC-lab/BUS3005-Resources/blob/main/Time_Series_Forecasting.ipynb)

A **time series** is data recorded over time at regular intervals &mdash; monthly sales, daily temperatures, hourly website visits. **Forecasting** means using the past values to predict future ones, which is one of the most practical skills in data science.

In this notebook we'll forecast **monthly electricity production** in the United States. Along the way we'll:

 * **explore** the data and pull it apart into *trend*, *seasonality*, and *noise*,
 * **forecast** future months with a classic statistical method, and
 * **evaluate** how accurate our forecast is.

Work through the cells top to bottom by clicking the **Play** icon beside each one.

> **A note on the code cells marked with ✨.** A few cells are left for *you* to complete using **Colab's built-in AI code generation**. Each one gives you a prompt to try &mdash; a nice way to practise steering an AI assistant.


# Getting the data (please read carefully)

Unlike a normal Python script, we'll **download the dataset by hand** and **upload it to Colab** &mdash; this is exactly how you'd work with your own data files.

### Step 1 &mdash; Download the CSV to your computer
Download the `Electric_Production.csv` from the LMS.

*(This is the well-known "Electric Production" dataset &mdash; a monthly index of US electricity production from 1985 to 2018. Original source: the [Kaggle Time Series datasets](https://www.kaggle.com/datasets/shenba/time-series-datasets) collection.)*

### Step 2 &mdash; Upload the CSV into this Colab session
 1. In the **left sidebar**, click the **folder icon** (📁 *Files*).
 2. Click the **Upload** button (a page with an up-arrow) &mdash; or simply **drag** `Electric_Production.csv` into that panel.
 3. Wait for the file to finish uploading. It will appear as `Electric_Production.csv` in the file list.

> ⚠️ **Important:** files you upload live in **temporary session storage**. When this Colab runtime disconnects or is recycled (e.g. after you close the tab or leave it idle), **the uploaded file is deleted**. If you come back later and see a "file not found" error, just **re-upload the CSV** using Step 2 again.

Once the file is uploaded, run the cells below.


First, our software libraries. These are all pre-installed in Colab, so this is quick.

In [ ]:
#@title Import software libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

print("Libraries ready.")

# Part 1 &mdash; Data Exploration

## Load the dataset and look at the first rows

We read the CSV from the session storage path `/content/`. If you get a `FileNotFoundError`, it means the upload didn't happen (or the session reset) &mdash; scroll up and repeat **Step 2**.


In [ ]:
CSV_PATH = "/content/Electric_Production.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        "Could not find Electric_Production.csv in this session.\n"
        "Upload it via the Files panel (folder icon on the left), then re-run this cell."
    )

df = pd.read_csv(CSV_PATH)
df.head()

The file has two columns: **`DATE`** and **`IPG2211A2N`**. That second name is just the official code for the electricity production index &mdash; we'll rename it to something friendlier.


## Convert the date column and set it as the index

Right now pandas treats `DATE` as plain text. For time-series work we need it to be a real **datetime**, and we set it as the DataFrame's **index** so that pandas knows the data is ordered in time. We also tell pandas the data is **monthly** (`MS` = month-start).


In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"])      # text -> real dates
df = df.set_index("DATE")                     # dates become the index
df = df.rename(columns={"IPG2211A2N": "Production"})
df = df.asfreq("MS")                          # state the monthly frequency

print(f"{len(df)} monthly observations, from {df.index.min().date()} to {df.index.max().date()}")
df.head()

## Visualise the time series

The first thing to do with any time series is simply **plot it** and look.


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df.index, df["Production"], color="#1f77b4")
plt.title("US Monthly Electricity Production Index (1985-2018)")
plt.xlabel("Year")
plt.ylabel("Production index")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Even at a glance you can see two things: the values **drift upward** over the decades, and they **wiggle up and down within each year** in a repeating pattern. Let's confirm both properly.


### ✨ Your turn (Colab AI code generation)

Let's make the trend and the year-to-year variation easier to see by overlaying a **rolling average**.

**How to use Colab's AI:** click inside the empty cell below, then press the **"Generate"** button (the ✨ *Generate with AI* option that appears in a code cell, or *Insert &rarr; Generate code*). Paste the prompt, run it, and check the result.


In [ ]:
# ✨ Use Colab's AI code generation in this cell.
# PROMPT to paste into the "Generate with AI" box:
#
#   "Plot the df['Production'] time series together with its 12-month rolling
#    mean and 12-month rolling standard deviation, with a legend and a title."
#
# (Then delete this comment and run the generated code.)


## Decompose the series into trend, seasonality, and residual

A powerful idea in time-series analysis is that a series can be **split into three parts**:

 * **Trend** &mdash; the slow, long-term direction (is it generally rising or falling?),
 * **Seasonality** &mdash; the pattern that repeats every fixed period (here, every 12 months), and
 * **Residual** &mdash; the leftover "noise" that trend and seasonality don't explain.

`seasonal_decompose` does this for us. We use `period=12` because the data is monthly and the pattern repeats yearly.


In [ ]:
decomposition = seasonal_decompose(df["Production"], model="additive", period=12)

fig = decomposition.plot()
fig.set_size_inches(11, 8)
fig.suptitle("Decomposition of electricity production", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## Describing the trend and seasonality

Reading the four panels above (original, trend, seasonal, residual):

 * **Trend:** there is a clear, steady **upward trend**. Electricity production rises from an index around 62 in the mid-1980s to over 100 by the 2010s &mdash; roughly a 60% increase &mdash; reflecting decades of economic and population growth. The climb is steepest through the 1990s and flattens a little after about 2008.

 * **Seasonality:** there is a strong, highly regular **yearly cycle**. Production **peaks twice** &mdash; in **summer** (July&ndash;August, air-conditioning) and **winter** (December&ndash;January, heating) &mdash; and dips in the milder **spring and autumn** (April&ndash;May and October). The size of this swing is large: about &plusmn;11 index points from the trend, and it stays remarkably consistent every year.

 * **Residual:** what's left over is small and looks fairly random, which tells us trend + seasonality already explain most of the behaviour &mdash; a good sign that this series should be **forecastable**.


# Part 2 &mdash; Forecasting

## Split into training and testing data

To check whether a forecast is any good, we hide the most recent data from the model, ask it to predict that period, and then compare against what really happened.

Crucially, with time series we **must split by time, not randomly** &mdash; we train on the earlier years and test on the most recent ones, mimicking how forecasting works in real life. We'll hold out the **last 24 months** as our test set.


In [ ]:
HORIZON = 24   # forecast the final 24 months

train = df.iloc[:-HORIZON]
test  = df.iloc[-HORIZON:]

print(f"Training on {len(train)} months: {train.index.min().date()} to {train.index.max().date()}")
print(f"Testing on  {len(test)} months: {test.index.min().date()} to {test.index.max().date()}")

## Apply a forecasting method: Holt-Winters

We'll use **Holt-Winters Exponential Smoothing**, a classic statistical forecasting method. It's a natural fit here because it explicitly models the two things we just found in the decomposition:

 * an **additive trend** (the steady climb), and
 * an **additive seasonal** cycle with a 12-month period (the yearly pattern).

It learns these patterns from the training data and extends them into the future.


In [ ]:
model = ExponentialSmoothing(
    train["Production"],
    trend="add",              # capture the upward trend
    seasonal="add",           # capture the yearly seasonal cycle
    seasonal_periods=12,      # 12 months in the cycle
).fit()

print("Model fitted.")

## Generate predictions

Now we forecast the 24 months of the test period and line them up against the real values.


In [ ]:
forecast = model.forecast(HORIZON)

comparison = pd.DataFrame({"actual": test["Production"], "forecast": forecast})
comparison.head(12)

## Plot the results

The clearest way to judge a forecast is to plot it against reality.


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.index[-60:], train["Production"].iloc[-60:], label="Training data", color="#1f77b4")
plt.plot(test.index, test["Production"], label="Actual", color="black", marker="o", markersize=3)
plt.plot(test.index, forecast, label="Forecast", color="red", linestyle="--", marker="s", markersize=3)
plt.axvline(test.index[0], color="grey", linestyle=":", label="Forecast starts")
plt.title("Electricity production: forecast vs. actual")
plt.xlabel("Year")
plt.ylabel("Production index")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Reading the plot:** the red dashed forecast follows the black actual line closely &mdash; it captures both the **summer and winter peaks** and the **spring/autumn dips**, and it sits at roughly the right level thanks to the trend. It isn't perfect (notice it slightly under-shoots a couple of the sharpest peaks), but for a simple model extending two years into the future with no new information, it tracks the real pattern remarkably well.


### ✨ Your turn (Colab AI code generation)

Holt-Winters isn't the only option. **SARIMA** (Seasonal ARIMA) is another classic seasonal forecasting method. Use Colab's AI to try it and compare.


In [ ]:
# ✨ Use Colab's AI code generation in this cell.
# PROMPT to paste into the "Generate with AI" box:
#
#   "Fit a SARIMA model from statsmodels on train['Production'] with a seasonal
#    period of 12, forecast the next 24 months, and plot the forecast against
#    test['Production']."
#
# (Then delete this comment and run the generated code.)


# Part 3 &mdash; Evaluation

## Calculate the Mean Absolute Error (MAE)

Numbers make the comparison precise. The **Mean Absolute Error (MAE)** is the average size of the forecast's mistakes, in the same units as the data. If MAE = 3, our forecast is off by about 3 index points per month on average. Lower is better.


In [ ]:
mae = mean_absolute_error(test["Production"], forecast)

print(f"Mean Absolute Error (MAE): {mae:.2f} index points")
print(f"Average production level:  {test['Production'].mean():.1f} index points")
print(f"So the forecast is off by about {mae / test['Production'].mean() * 100:.1f}% on average.")

### Is that good? Compare against a simple baseline

A number like "MAE = 3.3" only means something **relative to a baseline**. The simplest sensible forecast for seasonal data is the **seasonal naive** method: *"next January will equal last January."* If our model can't beat that, it isn't earning its keep.


In [ ]:
# Seasonal naive: predict each month using the value 12 months earlier
seasonal_naive = df["Production"].shift(12).iloc[-HORIZON:]
naive_mae = mean_absolute_error(test["Production"], seasonal_naive)

print(f"Holt-Winters MAE : {mae:.2f}")
print(f"Seasonal-naive MAE: {naive_mae:.2f}")
print("\nHolt-Winters is better!" if mae < naive_mae else "\nThe naive baseline wins here.")

### ✨ Your turn (Colab AI code generation)

MAE is one metric among several. Two other common ones are **RMSE** (which punishes big misses harder) and **MAPE** (which expresses error as a percentage). Use Colab's AI to add them.


In [ ]:
# ✨ Use Colab's AI code generation in this cell.
# PROMPT to paste into the "Generate with AI" box:
#
#   "Compute the RMSE and the MAPE (mean absolute percentage error) between
#    test['Production'] and the forecast, and print both with clear labels."
#
# (Then delete this comment and run the generated code.)


## Interpreting the accuracy and its limitations

**How accurate is it?** An MAE of roughly 3 index points on values that average around 100 is about a **3% error** &mdash; and our model beat the seasonal-naive baseline. For a simple, fast, interpretable method forecasting two years ahead, that's a solid result.

**But keep the limitations in mind:**

 * **The future may not resemble the past.** The model assumes the trend and seasonal pattern continue. A sudden shock &mdash; a recession, a pandemic, a policy change, a new technology &mdash; can break that assumption, and no amount of historical data will have warned it.
 * **Accuracy fades with distance.** Forecasting next month is far easier than forecasting two years out. Errors generally grow the further ahead you look.
 * **A point forecast hides uncertainty.** We drew a single red line, but real forecasts should also report a **range** (a confidence interval) so decision-makers know how much to trust it.
 * **One split isn't the whole story.** We tested on a single 24-month window. A fuller evaluation would roll the split forward across several periods to check the model is consistently good, not just lucky here.
 * **Simple models miss complex effects.** Holt-Winters knows only this one series. It can't use extra information (temperature, prices, holidays) that a richer model could exploit.


# Discussion points

 * Our forecast slightly under-shoots the sharpest peaks. Why might a smoothing method do that, and when would under-forecasting a peak actually matter (e.g. for a power company)?
 * We held out the last 24 months. How might the MAE change if we forecast only 6 months ahead instead? Try changing `HORIZON` and re-running.
 * The seasonal-naive baseline was surprisingly hard to beat. Why are simple baselines so important when evaluating any model?
 * What extra data would you add to improve an electricity forecast, and why?


# Further reading

 * [**Forecasting: Principles and Practice**](https://otexts.com/fpp3/) by Hyndman & Athanasopoulos &mdash; a superb, free online textbook on time-series forecasting.
 * [**statsmodels: Time Series Analysis**](https://www.statsmodels.org/stable/tsa.html) &mdash; documentation for the tools used here.
 * [**statsmodels: Exponential Smoothing**](https://www.statsmodels.org/stable/examples/notebooks/generated/exponential_smoothing.html) &mdash; more on the Holt-Winters method.

---
*This notebook uses the "Electric Production" dataset (monthly US electricity production index, 1985-2018), uploaded manually to Colab session storage, simplified for teaching.*
